In [2]:
import os
from pathlib import Path
import time
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

import torchvision
from torchvision import datasets, transforms

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

# ---- Paths & basic config ----
# Point this to your dataset folder structured as:
# data_dir/
#   train/class_x/*.jpg
#   train/class_y/*.jpg
#   val/class_x/*.jpg
#   val/class_y/*.jpg
data_dir = Path("D:\\Project\\Coconut Plantation\\Applicatno\\patches_split")  # <-- CHANGE ME
num_classes = 2            # <-- CHANGE ME
batch_size = 32
num_workers = 4
epochs = 15
lr = 3e-4
weight_decay = 1e-4
mix_precision = True  # turn off if you see instability
seed = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ---- Transforms (match your ResNet pipeline if needed) ----
input_size = 224  # MobileNet default input size
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(input_size),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize(int(input_size * 1.15)),
    transforms.CenterCrop(input_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ---- Datasets & Dataloaders ----
train_dir = data_dir / "train"
val_dir   = data_dir / "val"
assert train_dir.exists(), f"Missing folder: {train_dir}"
assert val_dir.exists(), f"Missing folder: {val_dir}"

train_dataset = datasets.ImageFolder(train_dir, transform=train_tfms)
val_dataset   = datasets.ImageFolder(val_dir, transform=val_tfms)

class_names = train_dataset.classes
if num_classes != len(class_names):
    print(f"[Note] num_classes={num_classes} but dataset has {len(class_names)} classes. "
          f"Overriding num_classes -> {len(class_names)} to match dataset.")
    num_classes = len(class_names)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=True,
)

val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=True,
)

len(train_dataset), len(val_dataset), class_names


Torch: 2.7.1+cpu
Torchvision: 0.22.1+cpu


AssertionError: Missing folder: D:\Project\Coconut Plantation\Applicatno\patches_split\val

In [ ]:
# ---- Model: MobileNetV2 ----
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

weights = MobileNet_V2_Weights.DEFAULT  # pretrained
model = mobilenet_v2(weights=weights)

# Replace classifier for our number of classes
# MobileNetV2 has model.classifier = Sequential(Dropout, Linear(1280, 1000))
in_feats = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_feats, num_classes)

model = model.to(device)
print(model.classifier)


In [ ]:
def evaluate(model, criterion, dataloader, device):
    model.eval()
    running_loss, running_corrects, n = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()
            n += inputs.size(0)
    epoch_loss = running_loss / n if n else 0.0
    epoch_acc  = running_corrects / n if n else 0.0
    return epoch_loss, epoch_acc

def train_model(model, criterion, optimizer, scheduler, train_loader, val_loader, device, epochs=10, mix_precision=True):
    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    scaler = GradScaler(enabled=mix_precision)

    for epoch in range(epochs):
        since = time.time()
        print(f"Epoch {epoch+1}/{epochs}")
        print("-"*30)
        # ---- Train ----
        model.train()
        running_loss, running_corrects, n = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=mix_precision):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()
            n += inputs.size(0)

        if scheduler is not None:
            scheduler.step()

        train_loss = running_loss / n if n else 0.0
        train_acc  = running_corrects / n if n else 0.0

        # ---- Validate ----
        val_loss, val_acc = evaluate(model, criterion, val_loader, device)

        time_elapsed = time.time() - since
        print(f"Train   - loss: {train_loss:.4f} | acc: {train_acc:.4f}")
        print(f"Val     - loss: {val_loss:.4f}  | acc: {val_acc:.4f}")
        print(f"Epoch time: {time_elapsed:.1f}s\n")

        # ---- Checkpoint ----
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
            torch.save({
                "model_state": best_wts,
                "acc": best_acc,
                "epoch": epoch+1,
            }, "best_model.pt")
            print(f"[Saved] New best model with acc={best_acc:.4f}")

    print(f"Best val acc: {best_acc:.4f}")
    model.load_state_dict(best_wts)
    return model


In [ ]:
# ---- Loss, Optimizer, Scheduler ----
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

# ---- Train ----
model = train_model(model, criterion, optimizer, scheduler, train_loader, val_loader, device, epochs=epochs, mix_precision=mix_precision)

# ---- Evaluate Once More ----
val_loss, val_acc = evaluate(model, criterion, val_loader, device)
print(f"Final Val - loss: {val_loss:.4f} | acc: {val_acc:.4f}")


In [ ]:
# ---- Optional: Export to ONNX ----
# Adjust a dummy input shape that matches your training (N, 3, 224, 224)
export_onnx = False
onnx_path = "mobilenet_export.onnx"

if export_onnx:
    model.eval()
    dummy = torch.randn(1, 3, 224, 224, device=device)
    torch.onnx.export(
        model, dummy, onnx_path,
        export_params=True, opset_version=13,
        do_constant_folding=True, input_names=["input"], output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}
    )
    print(f"Exported to {onnx_path}")
